In [20]:
import sys
import os
sys.path.append(os.path.abspath("..")) 
sys.path.append(os.path.abspath("../../source"))
from TDA_Testing import *
import json
import pandas as pd
import pickle
def savepkl(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
def loadpkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)
import gudhi as gd

# ============================================================
# 1. Construct X(b,d) on a circle
# ============================================================

def circle_point_cloud(b, d, center):
    """
    Construct X(b,d) on the circle of radius d.

    Adjacent points in cyclic order have Euclidean distance 2b,
    except possibly for the last-first pair, whose distance is <= 2b.
    """
    if not (0 < b < d):
        raise ValueError("Require 0 < b < d.")

    theta = 2.0 * np.arcsin(b / d)

    N = int(np.ceil(2.0 * np.pi / theta))

    angles = theta * np.arange(N)

    cx, cy = center

    X = np.column_stack([
        cx + d * np.cos(angles),
        cy + d * np.sin(angles)
    ])

    return X


# ============================================================
# 2. Two sufficiently separated circles
# ============================================================

def two_circle_point_cloud(b1, d1, b2, d2, L=10.0):

    center1 = (-L, 0.0)
    center2 = ( L, 0.0)

    X1 = circle_point_cloud(
        b=b1,
        d=d1,
        center=center1
    )

    X2 = circle_point_cloud(
        b=b2,
        d=d2,
        center=center2
    )

    X = np.vstack([X1, X2])

    return X


# ============================================================
# 3. Compute Cech H1 persistence
#    via Alpha complex
# ============================================================

def cech_h1_diagram(X):
    """
    Compute the H1 persistence diagram of the Cech filtration.

    AlphaComplex has the same persistent homology as the
    Euclidean Cech filtration.

    GUDHI stores squared radius values, so take square roots.
    """

    alpha = gd.AlphaComplex(points=X)

    st = alpha.create_simplex_tree()

    st.compute_persistence()

    H1_squared = st.persistence_intervals_in_dimension(1)

    # Convert squared filtration radius to radius
    H1 = np.sqrt(H1_squared)
    
    # Remove zero-persistence / diagonal intervals
    persistence = H1[:, 1] - H1[:, 0]
    tol = 1e-10
    H1 = H1[persistence > tol]


    return H1

def sample_F(rng):
    """
    Sample (b,d) from a fixed distribution F on Omega.
    """
    b = rng.uniform(0.15, 0.25)
    d = rng.uniform(0.45, 0.55)

    return b, d
    
def generate_diagrams_P(n=100, L=10.0, seed=42):
    """
    Generate n persistence diagrams from population P.

    P:
        Z ~ F
        Z1 = Z2 = Z

    Thus, the two circles have exactly the same (b,d)
    parameter within each observation.
    """
    rng = np.random.default_rng(seed)

    diagrams = []

    for _ in range(n):

        # One draw from F
        b, d = sample_F(rng)

        # Use the same (b,d) for both circles
        X = two_circle_point_cloud(
            b1=b,
            d1=d,
            b2=b,
            d2=d,
            L=L
        )

        # Compute Cech H1 persistence diagram
        D = cech_h1_diagram(X)

        diagrams.append(D)

    return diagrams


def generate_diagrams_Q(n=100, L=10.0, seed=123):
    """
    Generate n persistence diagrams from population Q.

    Q:
        Z1, Z2 iid ~ F

    Thus, the two circles have independent (b,d)
    parameters within each observation.
    """
    rng = np.random.default_rng(seed)

    diagrams = []

    for _ in range(n):

        # Two independent draws from F
        b1, d1 = sample_F(rng)
        b2, d2 = sample_F(rng)

        X = two_circle_point_cloud(
            b1=b1,
            d1=d1,
            b2=b2,
            d2=d2,
            L=L
        )

        # Compute Cech H1 persistence diagram
        D = cech_h1_diagram(X)

        diagrams.append(D)

    return diagrams

In [21]:
# simulation parameters
npc=50
nset=100
random.seed(42)
comb = np.array(list(combinations(range(nset), 2)))
ind=random.sample(range(len(comb)),nset)

#data generation 
P_diagrams = generate_diagrams_P(
    n=npc*nset,
    L=10.0,
    seed=42
)

Q_diagrams = generate_diagrams_Q(
    n=npc*nset,
    L=10.0,
    seed=123)


In [26]:
import rpy2.robjects as ro
from rpy2.robjects.vectors import FloatVector
def numpy_matrix_to_R(X):
    """
    Convert a NumPy matrix to an R matrix.
    """
    X = np.asarray(X, dtype=float)

    return ro.r.matrix(
        FloatVector(X.flatten(order="F")),
        nrow=X.shape[0],
        ncol=X.shape[1]
    )
def diagrams_to_R_list(diagrams):
    """
    Convert a Python list of persistence diagrams
    into an R list of matrices.
    """
    R_diagrams = [
        numpy_matrix_to_R(D)
        for D in diagrams
    ]

    return ro.r["list"](*R_diagrams)

P_diagrams_R = diagrams_to_R_list(P_diagrams)
Q_diagrams_R = diagrams_to_R_list(Q_diagrams)
ro.globalenv["P_diagrams"] = P_diagrams_R
ro.globalenv["Q_diagrams"] = Q_diagrams_R

ro.r('save(P_diagrams, Q_diagrams, file="../../data/Rdata/diagrams_for_equal_intensity_simul.RData")')

In [22]:
#linear_weight
random.seed(42)
func_weight = function_weight("Poly", poly_order=1)
Agg_linear_opt=np.zeros(nset)
for jj in range(nset):
    if jj%10==0:
        print(jj)
    X = P_diagrams[npc*(comb[ind[jj]][0]):npc*(comb[ind[jj]][0]+1)]
    Y = Q_diagrams[npc*(comb[ind[jj]][1]):npc*(comb[ind[jj]][1]+1)]
    test_result = Aggtest(X,Y,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True)
    Agg_linear_opt[jj] = test_result

0
10
20
30
40
50
60
70
80
90


In [32]:
print(np.sum(Agg_linear_opt)/nset)
savepkl(np.sum(Agg_linear_opt)/nset, '../../results/simulation_results/Equal_intensity_results/Equal_Agg_linear_opt.pkl')

0.04


In [27]:
#PD test
random.seed(42)
Perm=np.zeros(nset)
for jj in range(nset):
   X = P_diagrams[npc*(comb[ind[jj]][0]):npc*(comb[ind[jj]][0]+1)]
   Y = Q_diagrams[npc*(comb[ind[jj]][1]):npc*(comb[ind[jj]][1]+1)]
   T_obs, T_perm, p_val = permutation_test(X, Y, num_permutations=1000) 
   Perm[jj] = p_val

In [33]:
print(np.sum(Perm<0.05)/nset)
savepkl(np.sum(Perm<0.05)/nset, '../../results/simulation_results/Equal_intensity_results/Equal_PD_result.pkl')

1.0


In [45]:
# convert diagrams to PL
P_pllist=[]
Q_pllist=[]
for jj in range(len(P_diagrams)):
        P_pdmat = [
        np.empty((0, 2)),
        np.array(P_diagrams[jj])
        ]
        Q_pdmat = [
        np.empty((0, 2)),
        np.array(Q_diagrams[jj])
        ]
        P_pl = PersLandscapeApprox(dgms=P_pdmat, hom_deg=1) # compute persistence landscape
        Q_pl = PersLandscapeApprox(dgms=Q_pdmat, hom_deg=1) # compute persistence landscape
        P_pllist.append(P_pl)
        Q_pllist.append(Q_pl)

In [47]:
random.seed(42)
PL=np.zeros(nset)
for jj in range(nset):
   X = P_pllist[npc*(comb[ind[jj]][0]):npc*(comb[ind[jj]][0]+1)]
   Y = Q_pllist[npc*(comb[ind[jj]][1]):npc*(comb[ind[jj]][1]+1)]
   PL[jj] =permutation_pl_test(X ,Y) # pvalue 

In [48]:
print(np.sum(PL<0.05)/nset)
savepkl(np.sum(PL<0.05)/nset, '../../results/simulation_results/Equal_intensity_results/Equal_PL_result.pkl')

1.0
